# Notebook 13: Full MiniLM item embeddings on Colab GPU
Written by: Abhinita Sanabada

Prepared for Ruchika's first no-LLM control experiment. This notebook encodes **all 25,903 catalog items**. It does not generate user profiles, call a teacher LLM, train SASRec, or evaluate test metrics.

1. Select **Runtime → Change runtime type → A100 GPU** (another CUDA GPU also works).
2. Choose **Runtime → Run all**. Approve Google authentication and Drive access when prompted.
3. Wait for the final **PASS** report. Share the output folder or ZIP with Ruchika.

The MiniLM model and repository source use immutable commit IDs. GPU encoding uses FP32 and batch size 128. Outputs go to `MyDrive/298A/model1_item_text/` in a new timestamped folder. This prepared notebook has been syntax-checked locally; its Colab GPU execution is yours to run.


In [2]:
from pathlib import Path
REPO_URL = "https://github.com/Ashley96yy/data298a-team5-topic22.git"
REPO_COMMIT = "c73e35ea476131b22443ea6549094a05149828e6"
PROJECT = Path("/content/298A_minilm_pinned")
PROCESSED_FOLDER_ID = "1OZFOjwoZjgB2_JR9u2fkHxRBcZnsWm5G"
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
ENCODER = "minilm"
ENCODE_MODE = "full"
TRAIN_MODE = "off"
BATCH_SIZE = 128  # A100. Reduce to 32 if a smaller GPU runs out of memory.
EXPECTED_MANIFEST_SHA256 = "e4b7801c0d9ea27d0b39a6af253ac154d6f4a7181d8a7c1dc31d651a050af0e5"
EXPECTED_ENCODER_SOURCE_SHA256 = "c7e4a67d9ee68214719d8b965544fda143fe55b009236fc29a1709d5d1fe9239"


## 1. Load pinned code and dependencies
The checkout must match the recorded commit. Installation may take several minutes. If Colab explicitly requests a session restart, restart once and run the cells again.


In [3]:
import subprocess, sys, os, hashlib, json
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_COMMIT], cwd=PROJECT, check=True)
subprocess.run(["git", "checkout", "--detach", REPO_COMMIT], cwd=PROJECT, check=True)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
SOURCE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert SOURCE_COMMIT == REPO_COMMIT
assert hashlib.sha256((PROJECT / "src/data/item_embeddings.py").read_bytes()).hexdigest() == EXPECTED_ENCODER_SOURCE_SHA256
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-item-text.txt"], check=True)
print("Pinned source:", SOURCE_COMMIT)


Pinned source: c73e35ea476131b22443ea6549094a05149828e6


## 2. Authenticate and verify the frozen dataset
Sign in with the Google account that has access to the team's processed-data folder. The notebook downloads only the manifest, dataset information, and prepared item text. It verifies the manifest and both input files before encoding.


In [4]:
from google.colab import auth, drive
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import hashlib, json, re

auth.authenticate_user()
drive.mount("/content/drive")
credentials, _ = default()
service = build("drive", "v3", credentials=credentials, cache_discovery=False)
if not re.fullmatch(r"[A-Za-z0-9_-]+", PROCESSED_FOLDER_ID):
    raise ValueError("Invalid processed folder ID")
processed = PROJECT / "data/processed"
processed.mkdir(parents=True, exist_ok=True)
children_cache = {}
def child(folder_id, name):
    if folder_id not in children_cache:
        files, token = [], None
        while True:
            response = service.files().list(q=f"'{folder_id}' in parents and trashed = false",
                fields="nextPageToken,files(id,name,mimeType)", pageSize=1000, pageToken=token,
                supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            files.extend(response.get("files", []))
            token = response.get("nextPageToken")
            if not token: break
        children_cache[folder_id] = files
    matches = [f for f in children_cache[folder_id] if f['name'] == name]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {name!r} in folder {folder_id}; found {len(matches)}")
    return matches[0]['id']
def download(relative, expected=None):
    destination = processed / relative
    if destination.exists() and expected and hashlib.sha256(destination.read_bytes()).hexdigest() == expected:
        return
    file_id = PROCESSED_FOLDER_ID
    for name in Path(relative).parts:
        file_id = child(file_id, name)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".download")
    with temporary.open("wb") as output:
        downloader = MediaIoBaseDownload(output, service.files().get_media(fileId=file_id, supportsAllDrives=True))
        done = False
        while not done:
            _, done = downloader.next_chunk()
    if expected and hashlib.sha256(temporary.read_bytes()).hexdigest() != expected:
        temporary.unlink()
        raise ValueError(f"Frozen checksum mismatch: {relative}")
    temporary.replace(destination)
download("manifest.json")
assert hashlib.sha256((processed / "manifest.json").read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA256, "Frozen manifest changed. Confirm the dataset version before proceeding."
manifest = json.loads((processed / "manifest.json").read_text())
required = ["dataset_info.json", "item_text.parquet"]
if TRAIN_MODE != "off":
    required += ["rec_eval.py", "sequences.parquet", "eval/eval_protocol.json", "eval/neg_val.npy",
                 "eval/neg_test.npy", "eval/eval_cases.parquet", "groups/item_groups.parquet", "groups/user_groups.parquet"]
for name in required:
    download(name, manifest["processed_files"][name]["sha256"])
print("Verified files:", required)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Verified files: ['dataset_info.json', 'item_text.parquet']


## 3. Check the GPU and inputs
The matrix will have shape **(25,904, 384)**. Row zero is padding, and row `item_idx` belongs to that item. MiniLM uses attention-masked mean pooling, L2 normalization and a 256-token limit. This notebook does not use review snippets or held-out interaction text.


In [5]:
import importlib.metadata, platform, torch
from datetime import datetime, timezone
from src.data.item_embeddings import read_items, digest
assert ENCODE_MODE == "full" and TRAIN_MODE == "off"
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > A100 GPU, then rerun.")
GPU_NAME = torch.cuda.get_device_name(0)
EXPECTED_VERSIONS = {"torch": "2.8.0", "transformers": "4.57.1", "huggingface-hub": "0.36.0", "safetensors": "0.6.2", "numpy": "2.2.6", "pandas": "2.3.2", "pyarrow": "21.0.0"}
VERSIONS = {name: importlib.metadata.version(name) for name in EXPECTED_VERSIONS}
for name, expected in EXPECTED_VERSIONS.items():
    assert VERSIONS[name].split("+")[0] == expected, f"{name}: expected {expected}, found {VERSIONS[name]}"
assert torch.__version__.split("+")[0] == "2.8.0", "Restart the Colab session to load the installed torch version."
items, provenance = read_items(processed)
assert provenance["n_items"] == 25903
assert provenance["manifest_sha256"] == EXPECTED_MANIFEST_SHA256
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = Path("/content/drive/MyDrive/298A/model1_item_text") / f"minilm_full_pinned_{STAMP}"
assert not OUTPUT.exists()
print("GPU:", GPU_NAME)
print("Python:", platform.python_version(), "CUDA:", torch.version.cuda)
print("Versions:", VERSIONS)
print("Items:", len(items), "Batch size:", BATCH_SIZE)
print("Model:", MODEL_ID, "Revision:", MODEL_REVISION)
print("Save location:", OUTPUT)


GPU: Tesla T4
Python: 3.13.16 CUDA: 12.8
Versions: {'torch': '2.8.0', 'transformers': '4.57.1', 'huggingface-hub': '0.36.0', 'safetensors': '0.6.2', 'numpy': '2.2.6', 'pandas': '2.3.2', 'pyarrow': '21.0.0'}
Items: 25903 Batch size: 128
Model: sentence-transformers/all-MiniLM-L6-v2 Revision: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
Save location: /content/drive/MyDrive/298A/model1_item_text/minilm_full_pinned_20261112T002342156654Z


## 4. Encode the complete catalog
This performs the full MiniLM encoding only. The first run downloads the pinned public model weights. Progress prints every 1,024 items. Keep the runtime connected until the final verification cell finishes.


In [6]:
%pip uninstall -y torchvision torchaudio

In [7]:
import time
from src.data.item_embeddings import HFEncoder, encode_items, save_embeddings, ENCODERS
# Full FP32 inference. No mixed-precision autocast or teacher LLM.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
encoder = HFEncoder(MODEL_ID, MODEL_REVISION, ENCODER, "cuda", str(PROJECT / "results/encoder_cache"))
started_at = datetime.now(timezone.utc).isoformat()
torch.cuda.synchronize()
started = time.perf_counter()
class ProgressEncoder:
    def __init__(self, encoder):
        self.encoder = encoder
        self.count = 0
        self.next_report = 1024
    def __call__(self, texts):
        vectors = self.encoder(texts)
        self.count += len(texts)
        if self.count >= self.next_report or self.count == len(items):
            print(f"Encoded {self.count:,}/{len(items):,} items in {time.perf_counter() - started:.1f}s", flush=True)
            self.next_report = self.count + 1024
        return vectors
matrix, count = encode_items(items, provenance["n_items"], ProgressEncoder(encoder), 384, batch_size=BATCH_SIZE)
torch.cuda.synchronize()
metadata = {**provenance, **ENCODERS[ENCODER], "encoder": ENCODER,
    "model_id": MODEL_ID, "revision": MODEL_REVISION, "normalize": True,
    "encoded_items": count, "complete_catalog": count == provenance["n_items"],
    "batch_size": BATCH_SIZE, "device": "cuda", "gpu_name": GPU_NAME,
    "precision": "float32", "tf32": False, "cuda_version": torch.version.cuda,
    "seconds_encoding": time.perf_counter() - started,
    "started_at_utc": started_at, "finished_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_commit": SOURCE_COMMIT, "source_sha256": digest(PROJECT / "src/data/item_embeddings.py"),
    "versions": {"python": platform.python_version(), **VERSIONS},
    "training_run": False, "teacher_llm_used": False}
embedding_file = save_embeddings(OUTPUT, ENCODER, matrix, metadata)
print("Saved:", embedding_file)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Encoded 1,024/25,903 items in 3.3s
Encoded 2,048/25,903 items in 5.9s
Encoded 3,072/25,903 items in 8.5s
Encoded 4,096/25,903 items in 11.3s
Encoded 5,120/25,903 items in 14.6s
Encoded 6,144/25,903 items in 17.2s
Encoded 7,168/25,903 items in 19.9s
Encoded 8,192/25,903 items in 22.5s
Encoded 9,216/25,903 items in 25.3s
Encoded 10,240/25,903 items in 28.7s
Encoded 11,264/25,903 items in 31.4s
Encoded 12,288/25,903 items in 34.2s
Encoded 13,312/25,903 items in 36.9s
Encoded 14,336/25,903 items in 39.5s
Encoded 15,360/25,903 items in 42.9s
Encoded 16,384/25,903 items in 45.7s
Encoded 17,408/25,903 items in 48.3s
Encoded 18,432/25,903 items in 51.0s
Encoded 19,456/25,903 items in 54.2s
Encoded 20,480/25,903 items in 57.5s
Encoded 21,504/25,903 items in 60.4s
Encoded 22,528/25,903 items in 63.1s
Encoded 23,552/25,903 items in 65.8s
Encoded 24,576/25,903 items in 68.7s
Encoded 25,600/25,903 items in 72.2s
Encoded 25,903/25,903 items in 73.0s
Saved: /content/drive/MyDrive/298A/model1_item_tex

## 5. Verify and prepare the handoff
Share the entire output folder or the ZIP, including `metadata.json`. The verification checks the complete catalog, input/output hashes, finite normalized vectors, zero padding, and re-encoding of three item IDs. A PASS verifies embeddings, not a trained Model 1 result.


In [8]:
import numpy as np, zipfile
from src.data.item_embeddings import load_embeddings
loaded, saved = load_embeddings(OUTPUT, manifest_sha256=EXPECTED_MANIFEST_SHA256, n_items=25903)
assert loaded.shape == (25904, 384)
assert saved["revision"] == MODEL_REVISION and saved["model_id"] == MODEL_ID
assert saved["encoded_items"] == 25903 and saved["complete_catalog"] is True
norms = np.linalg.norm(loaded[1:], axis=1)
assert np.allclose(norms, 1.0, atol=1e-5), "Vectors are not normalized"
assert np.array_equal(loaded, matrix), "Saved embeddings differ from encoded values"
check_ids = np.array([1, 12952, 25903])
reencoded = encoder(items.iloc[check_ids - 1].item_text.tolist())
max_difference = float(np.max(np.abs(reencoded - loaded[check_ids])))
assert np.allclose(reencoded, loaded[check_ids], atol=1e-5), f"Item alignment check failed: {max_difference}"
verification = {"status": "PASS", "complete_catalog": True, "shape": list(loaded.shape),
    "dtype": str(loaded.dtype), "finite": bool(np.isfinite(loaded).all()),
    "padding_zero": bool(np.count_nonzero(loaded[0]) == 0),
    "unit_norm_min": float(norms.min()), "unit_norm_max": float(norms.max()),
    "row_ids_checked": check_ids.tolist(), "row_reencoding_max_abs_difference": max_difference,
    "embedding_sha256": digest(embedding_file), "metadata_sha256": digest(OUTPUT / "metadata.json"),
    "training_run": False, "test_evaluation_run": False}
(OUTPUT / "verification.json").write_text(json.dumps(verification, indent=2) + "\n")
(OUTPUT / "README.txt").write_text(
    "MiniLM item embeddings for the no-LLM control experiment.\n"
    "Prepared by: Abhinita Sanabada\n"
    f"Model: {MODEL_ID}\nRevision: {MODEL_REVISION}\nSource commit: {SOURCE_COMMIT}\n"
    "Matrix: float32 (25904, 384). Row 0 is zero padding; row item_idx matches the frozen item ID.\n"
    "All 25903 items encoded. Metadata-only item_text, mean pooling, L2 normalization, max 256 tokens.\n"
    "Includes input and output hashes. No recommender training, profile generation or test evaluation.\n"
    "Load with src.data.item_embeddings.load_embeddings and the matching frozen manifest.\n")
ARCHIVE = OUTPUT.parent / (OUTPUT.name + ".zip")
with zipfile.ZipFile(ARCHIVE, "x", zipfile.ZIP_DEFLATED) as archive:
    for name in ["item_text_emb_minilm.npy", "metadata.json", "verification.json", "README.txt"]:
        archive.write(OUTPUT / name, arcname=OUTPUT.name + "/" + name)
print(json.dumps(verification, indent=2))
print("PASS: full pinned MiniLM embeddings ready for Ruchika's control run.")
print("Share folder:", OUTPUT)
print("Or share ZIP:", ARCHIVE)
print("No SASRec training was started.")


{
  "status": "PASS",
  "complete_catalog": true,
  "shape": [
    25904,
    384
  ],
  "dtype": "float32",
  "finite": true,
  "padding_zero": true,
  "unit_norm_min": 0.9999998807907104,
  "unit_norm_max": 1.0000001192092896,
  "row_ids_checked": [
    1,
    12952,
    25903
  ],
  "row_reencoding_max_abs_difference": 7.82310962677002e-08,
  "embedding_sha256": "630aa65eb5edbb0482d96bc14c5e8d42af606fd284de214119c92e6c6c81082a",
  "metadata_sha256": "b58bc21c61947f8010ca4298a89d4a674571aa32b0d6a538c8ea8458c255eade",
  "training_run": false,
  "test_evaluation_run": false
}
PASS: full pinned MiniLM embeddings ready for Ruchika's control run.
Share folder: /content/drive/MyDrive/298A/model1_item_text/minilm_full_pinned_20261112T002342156654Z
Or share ZIP: /content/drive/MyDrive/298A/model1_item_text/minilm_full_pinned_20261112T002342156654Z.zip
No SASRec training was started.
